# Metodologia Design Science Research (DSR)

**Etapa de Pesquisa (Peffers et al., 2007):**
### 3. Design e Desenvolvimento (Design and Development)

**Objetivo Acadêmico:** Este notebook foca na ingestão e tratamento de dados não estruturados (cardápios). Na ótica do DSR, representa a inclusão de variáveis semânticas no artefato. A literatura moderna (KIM et al., 2023) destaca que a integração de dados descritivos de menus é um diferencial crítico para aumentar a acurácia em previsões de demanda de catering.


# 01c - Ingestão de Cardápio

Este notebook processa o arquivo de cardápio para preparação de processamento de linguagem natural.

In [1]:
import pandas as pd
import glob
import os
import unicodedata

# 1. Definições e Mapeamentos
MAPA_PROTEINA = {
    'feijoada': 'FEIJOADA', 
    'peixe': 'PEIXE',
    'pescada': 'PEIXE',
    'tilapia': 'PEIXE',
    'frango': 'AVE',
    'coxa': 'AVE',
    'sobrecoxa': 'AVE',
    'bisteca': 'SUINA',
    'lombo': 'SUINA',
    'pernil': 'SUINA',
    'suino': 'SUINA',
    'carne': 'BOVINA',
    'cupim': 'BOVINA',
    'bife': 'BOVINA',
    'almondegas': 'BOVINA',
    'pts': 'VEGETAL',
    'soja': 'VEGETAL',
    'grao de bico': 'VEGETAL',
    'omelete': 'OVO',
    'ovos': 'OVO',
}

MAPA_PREPARO = {
    'milanesa': 'FRITO',
    'frito': 'FRITO',
    'empanado': 'FRITO',
    'grelhad': 'GRELHADO',
    'assad': 'ASSADO',
    'churrasco': 'ASSADO',
    'parmegiana': 'FORNO',
    'escondidinho': 'FORNO',
    'ao sugo': 'MOLHO',
    'ao molho': 'MOLHO',
    'strogonoff': 'MOLHO',
    'cozido': 'COZIDO',
    'de panela': 'COZIDO',
}

def normalizar_texto(s):
    if not isinstance(s, str): return str(s)
    n = unicodedata.normalize('NFD', s).encode('ascii', 'ignore').decode('utf-8')
    return s.lower().strip()

def classificar_texto(texto, mapa, valor_padrao='OUTRO'):
    if pd.isna(texto) or str(texto).upper() == "NAN" or not texto: return valor_padrao
    texto_norm = str(texto).lower()
    for chave, categoria in mapa.items():
        if chave in texto_norm:
            return categoria
    return valor_padrao

# 2. Motor de Extração de Cardápio Dinâmico
def processar_excel_cardapio(caminho):
    try:
        df_bruto = pd.read_excel(caminho, header=None)
        extraidos = []
        linha_datas = -1
        linha_pp = -1
        linha_pv = -1
        
        for idx, row in df_bruto.iterrows():
            row_str = row.astype(str).str.upper()
            datas_conv = pd.to_datetime(row, errors='coerce')
            if datas_conv.notna().any() and linha_datas == -1:
                linha_datas = idx
            
            if any(row_str.str.contains("PRATO PRINCIPAL", na=False)):
                linha_pp = idx
            if any(row_str.str.contains("VEGETARIANO", na=False)):
                linha_pv = idx
                
            if linha_datas != -1 and linha_pp != -1 and linha_pv != -1:
                datas_na_linha = pd.to_datetime(df_bruto.iloc[linha_datas], errors='coerce')
                for col_idx, data_obj in datas_na_linha.items():
                    if pd.notna(data_obj) and not isinstance(data_obj, (int, float)):
                        pp = str(df_bruto.iloc[linha_pp, col_idx]).strip()
                        pv = str(df_bruto.iloc[linha_pv, col_idx]).strip()
                        pp = pp if pp.upper() != "NAN" else ""
                        pv = pv if pv.upper() != "NAN" else ""
                        if pp or pv:
                            extraidos.append({'data': data_obj, 'prato_principal': pp, 'prato_vegetariano': pv})
                linha_datas = -1
                linha_pp = -1
                linha_pv = -1
        return pd.DataFrame(extraidos)
    except Exception as e:
        print(f"❌ Erro em {os.path.basename(caminho)}: {e}")
        return pd.DataFrame()

# 3. Execução em Lote (Batch)
DIRETORIO_BRUTO = "../data/cardapio*/" # Busca flexível
lista_arquivos = glob.glob(os.path.join(DIRETORIO_BRUTO, "*.xlsx"))
if not lista_arquivos:
    lista_arquivos = glob.glob("../data/cardapios/*.xlsx")

print(f"🔍 Encontrados {len(lista_arquivos)} arquivos de cardápio.")

dfs = []
for f in lista_arquivos:
    print(f"📖 Lendo: {os.path.basename(f)}...")
    df_temp = processar_excel_cardapio(f)
    if not df_temp.empty:
        dfs.append(df_temp)

if dfs:
    df_final = pd.concat(dfs).drop_duplicates(subset=['data']).sort_values('data')
    df_final['proteina_principal'] = df_final['prato_principal'].apply(lambda x: classificar_texto(x, MAPA_PROTEINA, 'BOVINA'))
    df_final['preparo_principal'] = df_final['prato_principal'].apply(lambda x: classificar_texto(x, MAPA_PREPARO, 'COZIDO'))
    df_final['proteina_vegetariana'] = df_final['prato_vegetariano'].apply(lambda x: classificar_texto(x, MAPA_PROTEINA, 'VEGETAL'))
    
    # Exportação Final para a RAIZ de /data
    df_export = df_final[['data', 'proteina_principal', 'preparo_principal', 'proteina_vegetariana']].copy()
    saida = "../data/cardapio_consolidado.csv"
    df_export.to_csv(saida, index=False)
    print(f"🚀 SUCESSO! Base consolidada salva em: {saida}")
    # Sumário detalhado abaixo
else:
    print("⚠️ Nenhum arquivo de cardápio processado.")


🔍 Encontrados 8 arquivos de cardápio.
📖 Lendo: cardapio_abril.xlsx...


C:\Users\miche\AppData\Local\Temp\ipykernel_36576\3637146189.py:70: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datas_conv = pd.to_datetime(row, errors='coerce')
C:\Users\miche\AppData\Local\Temp\ipykernel_36576\3637146189.py:70: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datas_conv = pd.to_datetime(row, errors='coerce')
C:\Users\miche\AppData\Local\Temp\ipykernel_36576\3637146189.py:70: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datas_conv = pd.to_datetime(row, errors='coerce')
C:\Users\miche\AppData\Local\Temp\ipykernel_36576\3637146189.py:70: UserWarning: Could n

📖 Lendo: cardapio_agosto.xlsx...
📖 Lendo: cardapio_fevereiro.xlsx...
📖 Lendo: cardapio_geral.xlsx...
📖 Lendo: cardapio_julho.xlsx...


C:\Users\miche\AppData\Local\Temp\ipykernel_36576\3637146189.py:70: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datas_conv = pd.to_datetime(row, errors='coerce')
C:\Users\miche\AppData\Local\Temp\ipykernel_36576\3637146189.py:70: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datas_conv = pd.to_datetime(row, errors='coerce')
C:\Users\miche\AppData\Local\Temp\ipykernel_36576\3637146189.py:70: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datas_conv = pd.to_datetime(row, errors='coerce')
C:\Users\miche\AppData\Local\Temp\ipykernel_36576\3637146189.py:70: UserWarning: Could n

📖 Lendo: cardapio_junho.xlsx...
📖 Lendo: cardapio_maio.xlsx...
📖 Lendo: cardapio_março.xlsx...


🚀 SUCESSO! Base consolidada salva em: ../data/cardapio_consolidado.csv


# 7. Sumário da Ingestão de Cardápio
Resumo quantitativo e auditoria dos pratos processados.

In [2]:
# Sumário Detalhado (Auditoria e Detalhamento por Data)
import pandas as pd
df_temp = df_export.copy()
total_cardapios = len(df_temp)
total_dias = df_temp['data'].nunique()
primeira_data = df_temp['data'].min().strftime('%d/%m/%Y')
ultima_data = df_temp['data'].max().strftime('%d/%m/%Y')

print(f"--- RELATÓRIO DE INGESTÃO (CARDÁPIO) ---")
print(f"Total de Cardápios Extraídos: {total_cardapios}")
print(f"Total de Dias Cobertos: {total_dias}")
print(f"Período: {primeira_data} até {ultima_data}")

# 1. Auditoria de Proteínas
print("\nDistribuição por Proteína Principal:")
print(df_temp['proteina_principal'].value_counts().to_string())

print("\nDistribuição por Proteína Vegetariana:")
print(df_temp['proteina_vegetariana'].value_counts().to_string())

# 2. Visão Detalhada: Datas e Atributos Extraídos
print("\nDetalhamento por Data (Cronológico):")
detalhe_cardapio = df_temp[['data', 'proteina_principal', 'preparo_principal', 'proteina_vegetariana']].copy()
detalhe_cardapio.columns = ['Data', 'Proteína Principal', 'Preparo', 'Proteína Vegetariana']
detalhe_cardapio['Data'] = detalhe_cardapio['Data'].dt.strftime('%d/%m/%Y')

# Exibir tabela completa (o Jupyter cria barra de rolagem se for longa)
display(detalhe_cardapio)

# 3. Visão Mensal Rápida
df_temp['ano_mes'] = df_temp['data'].dt.strftime('%Y-%m')
res_por_mes = df_temp.groupby('ano_mes').size().reset_index(name='qtd')
print("\nVolume Consolidado por Mês:")
print(res_por_mes.to_string(index=False))

--- RELATÓRIO DE INGESTÃO (CARDÁPIO) ---
Total de Cardápios Extraídos: 125
Total de Dias Cobertos: 125
Período: 17/02/2025 até 29/08/2025

Distribuição por Proteína Principal:
proteina_principal
BOVINA      55
AVE         43
SUINA       17
FEIJOADA     5
PEIXE        5

Distribuição por Proteína Vegetariana:
proteina_vegetariana
VEGETAL     103
OVO          17
FEIJOADA      5

Detalhamento por Data (Cronológico):


,Data,Proteína Principal,Preparo,Proteína Vegetariana
0,17/02/2025,BOVINA,COZIDO,VEGETAL
1,18/02/2025,AVE,MOLHO,VEGETAL
2,19/02/2025,FEIJOADA,COZIDO,FEIJOADA
3,20/02/2025,AVE,COZIDO,VEGETAL
4,21/02/2025,BOVINA,COZIDO,VEGETAL
...,...,...,...,...
15,25/08/2025,BOVINA,MOLHO,VEGETAL
16,26/08/2025,AVE,COZIDO,VEGETAL
17,27/08/2025,BOVINA,COZIDO,VEGETAL
18,28/08/2025,BOVINA,COZIDO,VEGETAL



Volume Consolidado por Mês:
ano_mes  qtd
2025-02   10
2025-03   21
2025-04   22
2025-05   22
2025-06   21
2025-07    9
2025-08   20
